# Harness Ladder — suite_v2 on Kaggle T4/P100

Runs MiniCPM5-2B cumulative evals on `tasks/suite_v2.json` (64 tasks: GSM8K+BBH+MBPP).

**Order:** P8 (post-REPL-gate fix) → P2 → P4 → P6.

Enable **GPU** accelerator before Run All. Session may need ~4–8+ hours total; restart & resume from Working if needed (outputs saved under `/kaggle/working`).


In [ ]:
import os, sys, subprocess, torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), 'Turn on GPU accelerator (Settings → Accelerator → GPU)'


In [ ]:
# Clone tip of harness-ladder
REPO = 'https://github.com/ever-oli/harness-ladder.git'
ROOT = '/kaggle/working/harness-ladder'
if not os.path.exists(ROOT):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, ROOT])
else:
    subprocess.check_call(['git', '-C', ROOT, 'fetch', 'origin'])
    subprocess.check_call(['git', '-C', ROOT, 'reset', '--hard', 'origin/main'])
os.chdir(ROOT)
print(subprocess.check_output(['git', 'log', '-1', '--oneline'], text=True))
print('suite_v2', os.path.exists('tasks/suite_v2.json'))


In [ ]:
# Deps (Transformers path used by examples/run_p4_transformers.py)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.44.0', 'accelerate', 'sentencepiece', 'protobuf', 'einops'])
# repo requirements (light)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])


In [ ]:
import json
from pathlib import Path

def run_rung(rung: int, out_name: str, max_new_tokens: int = 160):
    out = Path('/kaggle/working') / out_name
    cmd = [
        sys.executable, 'examples/run_p4_transformers.py',
        '--rung', str(rung),
        '--suite', 'tasks/suite_v2.json',
        '--output', str(out),
        '--ledger', '/kaggle/working/ledger_suite_v2.csv',
        '--max-new-tokens', str(max_new_tokens),
    ]
    env = os.environ.copy()
    env['PYTHONPATH'] = str(Path(ROOT) / 'src')
    print('RUN', ' '.join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=ROOT, env=env)
    data = json.loads(out.read_text())
    print(f"P{rung} passed={data['n_success']}/{data['n_tasks']} rate={data['success_rate']:.2%}", flush=True)
    return data

# Priority: P8 fix check first
p8 = run_rung(8, 'p8_real_minicpm5_suite_v2_fix.json', max_new_tokens=192)
print('P8 done', p8['success_rate'])


In [ ]:
# Mid-rungs
p2 = run_rung(2, 'p2_real_minicpm5_suite_v2.json')
p4 = run_rung(4, 'p4_real_minicpm5_suite_v2.json')
p6 = run_rung(6, 'p6_real_minicpm5_suite_v2.json')
summary = {
    'P2': p2['success_rate'], 'P4': p4['success_rate'],
    'P6': p6['success_rate'], 'P8': p8['success_rate'],
    'passed': {
        'P2': f"{p2['n_success']}/{p2['n_tasks']}",
        'P4': f"{p4['n_success']}/{p4['n_tasks']}",
        'P6': f"{p6['n_success']}/{p6['n_tasks']}",
        'P8': f"{p8['n_success']}/{p8['n_tasks']}",
    }
}
Path('/kaggle/working/suite_v2_kaggle_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)


## Download

From the notebook: **Output** → download `/kaggle/working/*.json` and `ledger_suite_v2.csv`. Or File → Save Version → Save & Run All (commit outputs).
